## Import libraries

In [1]:
import os
from collections import Counter

import cv2
import numpy as np
from sklearn.utils import resample
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (BatchNormalization, Conv2D, Dense,
                                     Dropout, Flatten, Input, MaxPooling2D)
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

## Functions

In [2]:
def load_images_from_folders(base_path, folders):
    """Loads images from subfolders and converts them to grayscale."""
    x_data, y_data = [], []
    for label in folders:
        folder_path = os.path.join(base_path, str(label))
        if not os.path.exists(folder_path):
            print(f"Folder does not exist: {folder_path}")
            continue
        for fname in os.listdir(folder_path):
            img_path = os.path.join(folder_path, fname)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                img = cv2.resize(img, IMG_SIZE)
                x_data.append(img)
                y_data.append(label)
    return np.array(x_data), np.array(y_data)

def balance_dataset(x_data, y_data):
    """Balances the dataset using oversampling to match the majority class."""
    x_balanced, y_balanced = [], []
    counter = Counter(y_data)
    max_count = max(counter.values())
    for label in CLASSES:
        idx = np.where(y_data == label)[0]
        x_label = x_data[idx]
        y_label = y_data[idx]
        x_res, y_res = resample(x_label, y_label, replace=True,
                                n_samples=max_count, random_state=42)
        x_balanced.append(x_res)
        y_balanced.append(y_res)
    return np.vstack(x_balanced), np.hstack(y_balanced)

def prepare_data(x_data, y_data):
    """Normalizes images and prepares labels for Keras."""
    x_data = x_data.astype('float32') / 255.0
    x_data = np.expand_dims(x_data, -1)  # (N, 100, 100, 1)
    y_labels = y_data - 1  # Shift labels to 0-6 range
    y_cat = to_categorical(y_labels, num_classes=7)
    return x_data, y_cat

def build_cnn():
    """Defines a 4-block CNN architecture."""
    model = Sequential([
        Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1)),
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        # Block 4
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        # Classifier
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.4),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(7, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## Configuration & Constans

In [3]:
EMOTION_MAP = {
    1: 'Surprise', 2: 'Fear', 3: 'Disgust',
    4: 'Joy', 5: 'Sadness', 6: 'Anger', 7: 'Neutral'
}
CLASSES = sorted(EMOTION_MAP.keys())
IMG_SIZE = (100, 100)
SAVE_PATH = os.path.join('MODELS', 'INPUT - IMAGES')
TRAIN_NOMASK_PATH = os.path.join('DATA', 'IMAGES','WITHOUT MASK', 'train')
TRAIN_MASK_PATH = os.path.join('DATA', 'IMAGES', 'WITH MASK', 'train')
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)
x_train_nomask_raw, y_train_nomask_raw = load_images_from_folders(TRAIN_NOMASK_PATH, CLASSES)
x_train_mask_raw, y_train_mask_raw = load_images_from_folders(TRAIN_MASK_PATH, CLASSES)
x_nomask_bal, y_nomask_bal = balance_dataset(x_train_nomask_raw, y_train_nomask_raw)
x_mask_bal, y_mask_bal = balance_dataset(x_train_mask_raw, y_train_mask_raw)
training_scenarios = {
    "model_nomask_bal.h5": prepare_data(x_nomask_bal, y_nomask_bal),
    "model_mask_bal.h5": prepare_data(x_mask_bal, y_mask_bal)
}

## Trening data without mask

In [4]:
filename = "model_nomask_bal.h5"
x_train, y_train = prepare_data(x_nomask_bal, y_nomask_bal)
print(f"\n--- Training {filename} ---")
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
model = build_cnn()
model.fit(x_train, y_train, 
            epochs=50, 
            batch_size=32,
            validation_split=0.1,  
            callbacks=[early_stop],
            verbose=2)
save_full_path = os.path.join(SAVE_PATH, filename)
model.save(save_full_path)
print(f"Model saved to {save_full_path}")


--- Training model_nomask_bal.h5 ---
Epoch 1/50
907/907 - 232s - 256ms/step - accuracy: 0.2438 - loss: 1.9059 - val_accuracy: 0.0000e+00 - val_loss: 2.8234
Epoch 2/50
907/907 - 229s - 252ms/step - accuracy: 0.3709 - loss: 1.5560 - val_accuracy: 0.0000e+00 - val_loss: 2.7899
Epoch 3/50
907/907 - 228s - 252ms/step - accuracy: 0.4954 - loss: 1.2623 - val_accuracy: 0.0096 - val_loss: 2.5716
Epoch 4/50
907/907 - 229s - 252ms/step - accuracy: 0.5686 - loss: 1.0876 - val_accuracy: 0.0012 - val_loss: 2.5773
Epoch 5/50
907/907 - 230s - 254ms/step - accuracy: 0.6179 - loss: 0.9522 - val_accuracy: 0.0000e+00 - val_loss: 2.3913
Epoch 6/50
907/907 - 227s - 250ms/step - accuracy: 0.6875 - loss: 0.8319 - val_accuracy: 0.0561 - val_loss: 2.1213
Epoch 7/50
907/907 - 224s - 247ms/step - accuracy: 0.7748 - loss: 0.6725 - val_accuracy: 0.1104 - val_loss: 1.9356
Epoch 8/50
907/907 - 225s - 249ms/step - accuracy: 0.8145 - loss: 0.5604 - val_accuracy: 0.1259 - val_loss: 1.9399
Epoch 9/50
907/907 - 224s - 24

Model saved to MODELS\INPUT - IMAGES\model_nomask_bal.h5


## Trening data with mask

In [5]:
filename = "model_mask_bal.h5"
x_train, y_train = prepare_data(x_mask_bal, y_mask_bal)
print(f"\n--- Training {filename} ---")
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
model = build_cnn()
model.fit(x_train, y_train, 
            epochs=50, 
            batch_size=32,
            validation_split=0.1,  
            callbacks=[early_stop],
            verbose=2)
save_full_path = os.path.join(SAVE_PATH, filename)
model.save(save_full_path)
print(f"Model saved to {save_full_path}")


--- Training model_mask_bal.h5 ---
Epoch 1/50
907/907 - 225s - 248ms/step - accuracy: 0.1587 - loss: 1.9714 - val_accuracy: 0.0000e+00 - val_loss: 2.9974
Epoch 2/50
907/907 - 221s - 243ms/step - accuracy: 0.1568 - loss: 1.8993 - val_accuracy: 0.0000e+00 - val_loss: 3.0535
Epoch 3/50
907/907 - 222s - 244ms/step - accuracy: 0.1608 - loss: 1.8988 - val_accuracy: 0.0000e+00 - val_loss: 3.0680
Epoch 4/50
907/907 - 221s - 243ms/step - accuracy: 0.1600 - loss: 1.8989 - val_accuracy: 0.0000e+00 - val_loss: 3.0315
Epoch 5/50
907/907 - 223s - 246ms/step - accuracy: 0.1574 - loss: 1.8990 - val_accuracy: 0.0000e+00 - val_loss: 3.0397
Epoch 6/50
907/907 - 227s - 251ms/step - accuracy: 0.1583 - loss: 1.8991 - val_accuracy: 0.0000e+00 - val_loss: 3.1616


Model saved to MODELS\INPUT - IMAGES\model_mask_bal.h5
